# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All references to dataset components (record sets, fields, columns) use the Croissant schema `@id` to ensure clarity and reproducibility.

### Dataset Source
This dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`. This will fetch information about available record sets and fields, enabling downstream data analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")

## 2. Data Overview

List available record sets and their fields. All entities are referenced by their `@id` values. This provides a map of what tables and fields (columns) can be loaded for analysis.

Below, we print all `RecordSet` and their details using their `@id`, name, and first few fields, if available.

In [ ]:
# List all record sets (@id is used for all references)
record_sets = list(meta.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description.strip() if hasattr(rs, 'description') and rs.description else '(No description)'}")
    print(f"  Fields (@id): {[field.id for field in rs.fields]}")
    print(f"  Columns (@id): {[col.id for col in rs.columns] if hasattr(rs, 'columns') else 'N/A'}\n")

## 3. Data Extraction

Using the record set `@id`s from the overview above, we load data into Pandas DataFrames for analysis. 

**Note:** If more than one record set is present, all are loaded into a dictionary. Use the appropriate `@id` key for exploration.

In [ ]:
# Prepare to extract data for each record set (@id)

record_set_ids = [rs.id for rs in meta.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Load records as list of dicts
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows from RecordSet {rs_id}.")
        print(f"Columns: {list(df.columns)}\n")
    else:
        print(f"RecordSet {rs_id} yielded no records.\n")

# For demonstration, pick the first available (non-empty) record set
main_rs_id = next(iter(dataframes))
print(f"Using main record set: {main_rs_id}\nPreview:")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Below are typical processing steps: filtering records, normalizing numeric fields, and grouping by categorical fields. All columns are referenced by their Croissant `@id`.

In [ ]:
# Show available column @id's (schema-defined field IDs)
df = dataframes[main_rs_id]
print(f"Available columns (@id): {list(df.columns)}\n")

# Pick one numeric field and one group field (choose IDs from previous cell output)
# If uncertain, display value counts for each column to help select
sample_counts = df.apply(lambda x: pd.api.types.infer_dtype(x.dropna()))
print("Column types guess (mlcroissant returns column @id):")
print(sample_counts)

# Identify a numeric field by its @id (manually pick one likely to be numeric; update if needed)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No obvious numeric field -- please check dataset.")
else:
    print(f"Selected numeric field (@id): {numeric_field_id}")

# Pick a group/categorical field
group_field_id = None
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() > 1 and df[col].nunique() < len(df) / 2:
        group_field_id = col
        break
if group_field_id is None:
    print("No obvious categorical field -- please check dataset.")
else:
    print(f"Selected group field (@id): {group_field_id}")

# Example filter threshold -- use median if data present
if numeric_field_id:
    threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
    if threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold} (median): {len(filtered_df)} rows")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
            / filtered_df[numeric_field_id].std(ddof=0)
        )

        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by selected categorical field
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (mean of numeric columns):")
            print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field or analyze relationships between key variables. This uses the `@id` for field selection.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

if numeric_field_id:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=15, color='dodgerblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.grid(alpha=0.2)
    plt.show()
if numeric_field_id and group_field_id:
    plt.figure(figsize=(8,5))
    df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

- In this notebook, we used the Croissant `@id` identifiers for all manipulations and references, ensuring traceability and reproducibility when accessing FAIR biomedical datasets.
- We loaded and previewed all available record sets, extracted dataframes for further analysis, performed filtering, normalization, grouping, and basic visualization.
- All code is written to adapt dynamically to the schema; you can further extend this workflow for in-depth biomedical or statistical analyses.

**For more detail or questions, consult the Croissant schema and the [mlcroissant documentation](https://mlcommons.org/croissant/).**